# Temporal Fusion Transformer — Travel Time Model

A third model for the same per-stop travel-time regression task, implementing the
**Temporal Fusion Transformer** (Lim et al., 2019, *"Temporal Fusion Transformers for
Interpretable Multi-horizon Time Series Forecasting"*).

## What TFT adds over the previous two notebooks
Unlike the plain LSTM and the Conv-Attn-LSTM, which feed all features into the recurrent layer
undifferentiated, TFT explicitly separates inputs into three categories and gives the network
machinery to (a) learn *which variables matter, at each timestep* (Variable Selection Networks)
and (b) use trip-level context to condition everything downstream (static covariate encoders):

- **Static** — constant for the whole trip: `route_id`, `direction_id`, `shape_id`, `service_id`,
  `is_weekend`, `is_federal_holiday`, `is_school_day`, `has_major_event`, plus `month`/`weekday`
  (a trip's calendar date doesn't change mid-trip, so these are static too, not time-varying).
- **Known** (time-varying, knowable in advance) — schedule/geometry facts: `stop_sequence`,
  `trip_progress`, `hour`, `scheduled_arrival`, `scheduled_departure`, `scheduled_segment_time`,
  `stop_lat`, `stop_lon`, `bearing`, `segment_length`, `scheduled_segment_speed_mps`, `is_peak`.
- **Observed** (time-varying, only known as it happens) — live/real-time signals: vehicle
  `latitude`/`longitude`, weather (`temperature_c`, `precipitation_mm`, `snowfall_cm`,
  `windspeed_kmh`, `is_raining`, `is_snowing`, `is_fog`, `weathercode`), `upstream_delay_seconds`,
  `speed_mps`, `headway_seconds`, `ridership`, `transfers`.

## Architecture
```
static vars ─► Variable Selection Network ─► static embedding
                                                  │
                                    4 static-context GRNs
                     ┌───────────┬───────────────┼───────────────┐
                     ▼           ▼               ▼               ▼
              c_selection   c_seq_hidden   c_seq_cell      c_enrichment
                     │           │               │               │
known + observed     │           │               │               │
vars (per timestep) ─► Temporal VSN            LSTM init states   │
                        (context=c_selection)      │               │
                            │                      ▼               │
                            └──────────────►   LSTM encoder        │
                                                    │               │
                                          gate + residual + norm    │
                                                    │               │
                                          static enrichment GRN ◄───┘
                                                    │
                                  causal + padding masked self-attention
                                                    │
                                          gate + residual + norm
                                                    │
                                        position-wise feed-forward GRN
                                                    │
                                gate + residual + norm (skip to pre-attention state)
                                                    │
                                            output layer ─► per-stop travel time
```

**Deviations from the original paper** (worth stating explicitly in a research write-up):
- Point regression with masked MSE/MAE, not quantile forecasts — kept this way so results stay
  directly comparable to the LSTM and Conv-Attn-LSTM notebooks, which are also point estimators.
- Standard `nn.MultiheadAttention` rather than the paper's "interpretable" variant (shared value
  projection across heads). Functionally similar, but the per-head interpretability guarantee in
  the original paper doesn't strictly hold here.
- Single causal encoder over the whole trip (predict at every stop using only same-or-earlier
  info) rather than the paper's separate encoder/decoder split for a known future horizon — this
  problem doesn't have a "future" horizon beyond the trip itself, so it collapses naturally into
  that shape.

Everything else — data loading, GroupShuffleSplit, encoding/normalization fit on train only,
masked loss, evaluation metrics, seeds — mirrors the earlier notebooks exactly for comparability.


In [1]:
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# =====================================================================
# 0. Config
# =====================================================================
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"
TARGET = "travel_time"
SEQ_ID_COLS = ["trip_id", "start_date"]
ORDER_COL = "stop_sequence"

# --- feature taxonomy (see markdown above for the reasoning) ---
STATIC_CATEGORICAL = ["route_id", "direction_id", "shape_id", "service_id",
                       "is_weekend", "is_federal_holiday", "is_school_day", "has_major_event"]
STATIC_NUMERIC = ["month", "weekday"]

KNOWN_CATEGORICAL = ["is_peak"]
KNOWN_NUMERIC = ["stop_sequence", "trip_progress", "hour",
                  "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
                  "stop_lat", "stop_lon", "bearing",
                  "segment_length", "scheduled_segment_speed_mps"]

OBSERVED_CATEGORICAL = ["is_raining", "is_snowing", "is_fog", "weathercode"]
OBSERVED_NUMERIC = ["latitude", "longitude",
                     "temperature_c", "precipitation_mm", "snowfall_cm", "windspeed_kmh",
                     "upstream_delay_seconds", "speed_mps", "headway_seconds",
                     "ridership", "transfers"]

CATEGORICAL = STATIC_CATEGORICAL + KNOWN_CATEGORICAL + OBSERVED_CATEGORICAL
NUMERIC = STATIC_NUMERIC + KNOWN_NUMERIC + OBSERVED_NUMERIC

# --- model hyperparameters ---
HIDDEN_SIZE = 128       # d_model used throughout TFT (variable embeddings, LSTM, attention)
N_HEADS = 4
N_LSTM_LAYERS = 1
DROPOUT = 0.1

BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 100
PATIENCE = 12
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)


In [2]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.shape}")

df = df.with_columns(
    (pl.col("trip_id") + "_" + pl.col("start_date").cast(pl.Utf8)).alias("_seq_id")
)

groups = df["_seq_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df[train_idx]
test_df = df[test_idx]
print(f"train rows: {train_df.height:,} | test rows: {test_df.height:,}")
print(f"train trips: {train_df['_seq_id'].n_unique():,} | test trips: {test_df['_seq_id'].n_unique():,}")


Loaded (5596768, 40)
train rows: 4,478,560 | test rows: 1,118,208
train trips: 96,308 | test trips: 24,077


In [3]:
# =====================================================================
# 2. Encode categoricals — fit on TRAIN only, reserve 0 for padding/unknown
# =====================================================================
cat_maps = {}
cat_cardinalities = {}   # dict: column name -> cardinality (TFT needs per-name lookup, not a list)
for col in CATEGORICAL:
    uniques = train_df[col].unique().sort().to_list()
    cat_maps[col] = {v: i + 1 for i, v in enumerate(uniques)}
    cat_cardinalities[col] = len(uniques) + 1


def encode_categoricals(d: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col in CATEGORICAL:
        mapping = cat_maps[col]
        exprs.append(
            pl.col(col).cast(pl.Utf8).replace_strict(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")
        )
    return d.with_columns(exprs)


train_df = encode_categoricals(train_df)
test_df = encode_categoricals(test_df)

STATIC_CAT_CODE_COLS = [f"_{c}_code" for c in STATIC_CATEGORICAL]
KNOWN_CAT_CODE_COLS = [f"_{c}_code" for c in KNOWN_CATEGORICAL]
OBSERVED_CAT_CODE_COLS = [f"_{c}_code" for c in OBSERVED_CATEGORICAL]


In [4]:
# =====================================================================
# 3. Normalize numerics — fit mean/std on TRAIN only
# =====================================================================
num_means = {c: train_df[c].mean() for c in NUMERIC}
num_stds = {c: (train_df[c].std() or 1.0) for c in NUMERIC}
num_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in num_stds.items()}


def normalize_numeric(d: pl.DataFrame) -> pl.DataFrame:
    return d.with_columns([
        ((pl.col(c) - num_means[c]) / num_stds[c]).alias(c) for c in NUMERIC
    ])


train_df = normalize_numeric(train_df)
test_df = normalize_numeric(test_df)


In [5]:
# =====================================================================
# 4. Build one sequence per trip, split into static / known / observed arrays
# =====================================================================
def build_sequences(d: pl.DataFrame) -> list[dict]:
    d = d.sort(SEQ_ID_COLS + [ORDER_COL])
    sequences = []
    for _, group in d.group_by("_seq_id", maintain_order=True):
        static_cat = group.select(STATIC_CAT_CODE_COLS).to_numpy()[0]                    # (n_static_cat,)
        static_num = group.select(STATIC_NUMERIC).to_numpy()[0].astype(np.float32)       # (n_static_num,)
        known_cat = group.select(KNOWN_CAT_CODE_COLS).to_numpy()                         # (T, n_known_cat)
        known_num = group.select(KNOWN_NUMERIC).to_numpy().astype(np.float32)            # (T, n_known_num)
        obs_cat = group.select(OBSERVED_CAT_CODE_COLS).to_numpy()                        # (T, n_obs_cat)
        obs_num = group.select(OBSERVED_NUMERIC).to_numpy().astype(np.float32)           # (T, n_obs_num)
        target = group[TARGET].to_numpy().astype(np.float32)
        sequences.append(dict(
            static_cat=static_cat, static_num=static_num,
            known_cat=known_cat, known_num=known_num,
            obs_cat=obs_cat, obs_num=obs_num,
            target=target, length=len(target),
        ))
    return sequences


print("Building train sequences...")
train_sequences = build_sequences(train_df)
print("Building test sequences...")
test_sequences = build_sequences(test_df)
print(f"train sequences: {len(train_sequences):,} | test sequences: {len(test_sequences):,}")


Building train sequences...
Building test sequences...
train sequences: 96,308 | test sequences: 24,077


In [6]:
# =====================================================================
# 5. Dataset / collate
# =====================================================================
class TripSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def collate(batch):
    static_cat = torch.tensor(np.stack([b["static_cat"] for b in batch]), dtype=torch.long)
    static_num = torch.tensor(np.stack([b["static_num"] for b in batch]), dtype=torch.float32)

    known_cat = pad_sequence([torch.tensor(b["known_cat"], dtype=torch.long) for b in batch], batch_first=True)
    known_num = pad_sequence([torch.tensor(b["known_num"], dtype=torch.float32) for b in batch], batch_first=True)
    obs_cat = pad_sequence([torch.tensor(b["obs_cat"], dtype=torch.long) for b in batch], batch_first=True)
    obs_num = pad_sequence([torch.tensor(b["obs_num"], dtype=torch.float32) for b in batch], batch_first=True)
    target = pad_sequence([torch.tensor(b["target"], dtype=torch.float32) for b in batch], batch_first=True)
    lengths = torch.tensor([b["length"] for b in batch], dtype=torch.long)

    return static_cat, static_num, known_cat, known_num, obs_cat, obs_num, target, lengths


train_loader = DataLoader(TripSequenceDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TripSequenceDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


## 6. TFT building blocks — GLU, Gated Residual Network, Variable Selection Network

These three modules are the reusable machinery the TFT paper is built from.

In [7]:
class GLU(nn.Module):
    """Gated Linear Unit: learns how much of its input to pass through, per channel."""
    def __init__(self, input_size, output_size=None):
        super().__init__()
        output_size = output_size or input_size
        self.fc = nn.Linear(input_size, output_size * 2)

    def forward(self, x):
        a, b = self.fc(x).chunk(2, dim=-1)
        return a * torch.sigmoid(b)


class GRN(nn.Module):
    """Gated Residual Network — the core TFT building block.

    y = LayerNorm(skip(x) + GLU(W2 . ELU(W1.x + W_context.context) ))
    Used everywhere: per-variable transforms, static context vectors, static enrichment,
    the position-wise feed-forward layer.
    """
    def __init__(self, input_size, hidden_size, output_size=None, context_size=None, dropout=0.1):
        super().__init__()
        output_size = output_size or input_size
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.context_fc = nn.Linear(context_size, hidden_size, bias=False) if context_size else None
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)
        self.glu = GLU(output_size, output_size)
        self.norm = nn.LayerNorm(output_size)

    def forward(self, x, context=None):
        residual = self.skip(x)
        h = self.fc1(x)
        if context is not None and self.context_fc is not None:
            h = h + self.context_fc(context)
        h = self.fc2(self.elu(h))
        h = self.glu(self.dropout(h))
        return self.norm(residual + h)


class VariableSelectionNetwork(nn.Module):
    """Learns a softmax weight per input variable (optionally conditioned on a context
    vector) and returns the weighted combination of each variable's own GRN-transformed
    representation. This is what gives TFT per-timestep, per-variable interpretability.
    """
    def __init__(self, var_dims: dict, hidden_size, context_size=None, dropout=0.1):
        super().__init__()
        self.names = list(var_dims.keys())
        self.hidden_size = hidden_size
        self.var_grns = nn.ModuleDict({
            name: GRN(dim, hidden_size, hidden_size, dropout=dropout) for name, dim in var_dims.items()
        })
        flattened_dim = sum(var_dims.values())
        self.flattened_grn = GRN(flattened_dim, hidden_size, len(self.names), context_size=context_size, dropout=dropout)

    def forward(self, inputs: dict, context=None):
        flat = torch.cat([inputs[n] for n in self.names], dim=-1)
        weights = torch.softmax(self.flattened_grn(flat, context), dim=-1).unsqueeze(-1)   # (..., n_vars, 1)
        transformed = torch.stack([self.var_grns[n](inputs[n]) for n in self.names], dim=-2)  # (..., n_vars, hidden)
        combined = (transformed * weights).sum(dim=-2)
        return combined, weights.squeeze(-1)


## 7. Temporal Fusion Transformer model

In [8]:
class TemporalFusionTransformer(nn.Module):
    def __init__(self, cat_cardinalities: dict,
                 static_cat_cols, static_num_cols, known_cat_cols, known_num_cols,
                 obs_cat_cols, obs_num_cols,
                 hidden_size, n_heads, n_lstm_layers, dropout):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_lstm_layers = n_lstm_layers
        self.static_cat_cols, self.static_num_cols = static_cat_cols, static_num_cols
        self.known_cat_cols, self.known_num_cols = known_cat_cols, known_num_cols
        self.obs_cat_cols, self.obs_num_cols = obs_cat_cols, obs_num_cols

        # every variable — categorical or numeric, static or time-varying — is projected
        # to the same hidden_size ("d_model") before variable selection, per the paper.
        all_cat_cols = static_cat_cols + known_cat_cols + obs_cat_cols
        all_num_cols = static_num_cols + known_num_cols + obs_num_cols
        self.cat_embeddings = nn.ModuleDict({
            col: nn.Embedding(cat_cardinalities[col], hidden_size, padding_idx=0) for col in all_cat_cols
        })
        self.num_projections = nn.ModuleDict({
            col: nn.Linear(1, hidden_size) for col in all_num_cols
        })

        static_dims = {c: hidden_size for c in static_cat_cols + static_num_cols}
        temporal_dims = {c: hidden_size for c in known_cat_cols + known_num_cols + obs_cat_cols + obs_num_cols}

        self.static_vsn = VariableSelectionNetwork(static_dims, hidden_size, context_size=None, dropout=dropout)
        self.temporal_vsn = VariableSelectionNetwork(temporal_dims, hidden_size, context_size=hidden_size, dropout=dropout)

        # the four static-context GRNs from the paper (Sec. 4.3)
        self.context_selection = GRN(hidden_size, hidden_size, hidden_size, dropout=dropout)
        self.context_seq_hidden = GRN(hidden_size, hidden_size, hidden_size, dropout=dropout)
        self.context_seq_cell = GRN(hidden_size, hidden_size, hidden_size, dropout=dropout)
        self.context_enrichment = GRN(hidden_size, hidden_size, hidden_size, dropout=dropout)

        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers, batch_first=True,
                             dropout=dropout if n_lstm_layers > 1 else 0.0)

        self.post_lstm_gate = GLU(hidden_size, hidden_size)
        self.post_lstm_norm = nn.LayerNorm(hidden_size)

        self.static_enrichment = GRN(hidden_size, hidden_size, hidden_size, context_size=hidden_size, dropout=dropout)

        self.attn = nn.MultiheadAttention(hidden_size, n_heads, dropout=dropout, batch_first=True)
        self.post_attn_gate = GLU(hidden_size, hidden_size)
        self.post_attn_norm = nn.LayerNorm(hidden_size)

        self.position_wise_ffn = GRN(hidden_size, hidden_size, hidden_size, dropout=dropout)
        self.final_gate = GLU(hidden_size, hidden_size)
        self.final_norm = nn.LayerNorm(hidden_size)

        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, static_cat, static_num, known_cat, known_num, obs_cat, obs_num, lengths,
                return_weights=False):
        B = static_cat.size(0)
        T = known_num.size(1)
        device = known_num.device

        # ---- static branch ----
        static_inputs = {}
        for i, col in enumerate(self.static_cat_cols):
            static_inputs[col] = self.cat_embeddings[col](static_cat[:, i])
        for i, col in enumerate(self.static_num_cols):
            static_inputs[col] = self.num_projections[col](static_num[:, i:i + 1])

        static_embedding, static_weights = self.static_vsn(static_inputs)   # (B, hidden), (B, n_static_vars)

        c_selection = self.context_selection(static_embedding)
        c_seq_hidden = self.context_seq_hidden(static_embedding)
        c_seq_cell = self.context_seq_cell(static_embedding)
        c_enrichment = self.context_enrichment(static_embedding)

        # ---- temporal branch ----
        temporal_inputs = {}
        for i, col in enumerate(self.known_cat_cols):
            temporal_inputs[col] = self.cat_embeddings[col](known_cat[:, :, i])
        for i, col in enumerate(self.known_num_cols):
            temporal_inputs[col] = self.num_projections[col](known_num[:, :, i:i + 1])
        for i, col in enumerate(self.obs_cat_cols):
            temporal_inputs[col] = self.cat_embeddings[col](obs_cat[:, :, i])
        for i, col in enumerate(self.obs_num_cols):
            temporal_inputs[col] = self.num_projections[col](obs_num[:, :, i:i + 1])

        selection_context = c_selection.unsqueeze(1).expand(B, T, self.hidden_size)
        temporal_embedding, temporal_weights = self.temporal_vsn(temporal_inputs, selection_context)  # (B,T,hidden)

        # ---- LSTM encoder, state initialized from static context ----
        h0 = c_seq_hidden.unsqueeze(0).expand(self.n_lstm_layers, B, self.hidden_size).contiguous()
        c0 = c_seq_cell.unsqueeze(0).expand(self.n_lstm_layers, B, self.hidden_size).contiguous()

        packed = pack_padded_sequence(temporal_embedding, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed, (h0, c0))
        lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)

        gated_lstm = self.post_lstm_norm(temporal_embedding + self.post_lstm_gate(lstm_out))

        # ---- static enrichment ----
        enrichment_context = c_enrichment.unsqueeze(1).expand(B, T, self.hidden_size)
        enriched = self.static_enrichment(gated_lstm, enrichment_context)

        # ---- causal + padding masked self-attention ----
        causal_mask = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
        key_padding_mask = torch.arange(T, device=device).unsqueeze(0) >= lengths.unsqueeze(1).to(device)
        attn_out, attn_weights = self.attn(
            enriched, enriched, enriched,
            attn_mask=causal_mask, key_padding_mask=key_padding_mask,
            need_weights=return_weights,
        )
        gated_attn = self.post_attn_norm(enriched + self.post_attn_gate(attn_out))

        ffn_out = self.position_wise_ffn(gated_attn)
        # final skip connects all the way back to the pre-enrichment/attention signal, so the
        # network can learn to bypass attention entirely if it isn't useful for a given input.
        out = self.final_norm(gated_lstm + self.final_gate(ffn_out))

        pred = self.output_layer(out).squeeze(-1)

        if return_weights:
            return pred, static_weights, temporal_weights, attn_weights
        return pred


def masked_mae(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (torch.abs(pred - target) * mask).sum() / mask.sum().clamp(min=1)


def masked_mse(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1)


model = TemporalFusionTransformer(
    cat_cardinalities,
    STATIC_CATEGORICAL, STATIC_NUMERIC, KNOWN_CATEGORICAL, KNOWN_NUMERIC,
    OBSERVED_CATEGORICAL, OBSERVED_NUMERIC,
    HIDDEN_SIZE, N_HEADS, N_LSTM_LAYERS, DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=MAX_EPOCHS, steps_per_epoch=len(train_loader),
)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")


TemporalFusionTransformer(
  (cat_embeddings): ModuleDict(
    (route_id): Embedding(6, 128, padding_idx=0)
    (direction_id): Embedding(3, 128, padding_idx=0)
    (shape_id): Embedding(79, 128, padding_idx=0)
    (service_id): Embedding(49, 128, padding_idx=0)
    (is_weekend): Embedding(3, 128, padding_idx=0)
    (is_federal_holiday): Embedding(3, 128, padding_idx=0)
    (is_school_day): Embedding(3, 128, padding_idx=0)
    (has_major_event): Embedding(3, 128, padding_idx=0)
    (is_peak): Embedding(3, 128, padding_idx=0)
    (is_raining): Embedding(3, 128, padding_idx=0)
    (is_snowing): Embedding(3, 128, padding_idx=0)
    (is_fog): Embedding(2, 128, padding_idx=0)
    (weathercode): Embedding(14, 128, padding_idx=0)
  )
  (num_projections): ModuleDict(
    (month): Linear(in_features=1, out_features=128, bias=True)
    (weekday): Linear(in_features=1, out_features=128, bias=True)
    (stop_sequence): Linear(in_features=1, out_features=128, bias=True)
    (trip_progress): Linear(

In [9]:
# =====================================================================
# 8. Train loop with early stopping on validation (test) MAE
# =====================================================================
def run_epoch(loader, train: bool) -> float:
    model.train() if train else model.eval()
    total_mae, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for static_cat, static_num, known_cat, known_num, obs_cat, obs_num, target, lengths in loader:
            static_cat, static_num = static_cat.to(DEVICE), static_num.to(DEVICE)
            known_cat, known_num = known_cat.to(DEVICE), known_num.to(DEVICE)
            obs_cat, obs_num = obs_cat.to(DEVICE), obs_num.to(DEVICE)
            target = target.to(DEVICE)

            pred = model(static_cat, static_num, known_cat, known_num, obs_cat, obs_num, lengths)
            loss = masked_mse(pred, target, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()
            mae = masked_mae(pred, target, lengths)
            n = lengths.sum().item()
            total_mae += mae.item() * n
            total_count += n
    return total_mae / total_count


best_val_mae = float("inf")
epochs_no_improve = 0
best_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    train_mae = run_epoch(train_loader, train=True)
    val_mae = run_epoch(test_loader, train=False)
    print(f"epoch {epoch:3d} | train MAE {train_mae:7.2f}s | val MAE {val_mae:7.2f}s")

    if val_mae < best_val_mae - 1e-3:
        best_val_mae = val_mae
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MAE {best_val_mae:.2f}s)")
            break

model.load_state_dict(best_state)


epoch   1 | train MAE   71.17s | val MAE   54.92s
epoch   2 | train MAE   44.91s | val MAE   37.59s
epoch   3 | train MAE   34.54s | val MAE   32.61s
epoch   4 | train MAE   31.84s | val MAE   31.27s
epoch   5 | train MAE   30.91s | val MAE   30.53s
epoch   6 | train MAE   30.53s | val MAE   30.22s
epoch   7 | train MAE   30.32s | val MAE   30.23s
epoch   8 | train MAE   30.21s | val MAE   30.17s
epoch   9 | train MAE   30.12s | val MAE   29.71s
epoch  10 | train MAE   30.03s | val MAE   30.13s
epoch  11 | train MAE   29.95s | val MAE   30.09s
epoch  12 | train MAE   29.92s | val MAE   29.99s
epoch  13 | train MAE   29.88s | val MAE   29.63s
epoch  14 | train MAE   29.86s | val MAE   29.86s
epoch  15 | train MAE   29.82s | val MAE   30.36s
epoch  16 | train MAE   29.81s | val MAE   29.81s
epoch  17 | train MAE   29.79s | val MAE   29.64s
epoch  18 | train MAE   29.80s | val MAE   29.91s
epoch  19 | train MAE   29.79s | val MAE   29.83s
epoch  20 | train MAE   29.79s | val MAE   29.94s


<All keys matched successfully>

In [10]:
# =====================================================================
# 9. Final evaluation — same metrics as the other notebooks, directly comparable
# =====================================================================
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for static_cat, static_num, known_cat, known_num, obs_cat, obs_num, target, lengths in test_loader:
        static_cat, static_num = static_cat.to(DEVICE), static_num.to(DEVICE)
        known_cat, known_num = known_cat.to(DEVICE), known_num.to(DEVICE)
        obs_cat, obs_num = obs_cat.to(DEVICE), obs_num.to(DEVICE)
        pred = model(static_cat, static_num, known_cat, known_num, obs_cat, obs_num, lengths).cpu()
        for i, length in enumerate(lengths):
            all_preds.append(pred[i, :length].numpy())
            all_targets.append(target[i, :length].numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
resid = preds - targets

mae = np.abs(resid).mean()
rmse = np.sqrt((resid ** 2).mean())
median_ae = np.median(np.abs(resid))
bias = resid.mean()
ss_res = (resid ** 2).sum()
ss_tot = ((targets - targets.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

print(f"\nMAE:       {mae:.1f} sec")
print(f"Median AE: {median_ae:.1f} sec")
print(f"RMSE:      {rmse:.1f} sec")
print(f"Bias:      {bias:+.1f} sec")
print(f"R2:        {r2:.4f}")
for thresh in (30, 60, 120):
    print(f"within {thresh}s: {(np.abs(resid) <= thresh).mean():.1%}")

import os
os.makedirs("models", exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "cat_maps": cat_maps,
        "num_means": num_means,
        "num_stds": num_stds,
        "config": {
            "HIDDEN_SIZE": HIDDEN_SIZE, "N_HEADS": N_HEADS, "N_LSTM_LAYERS": N_LSTM_LAYERS, "DROPOUT": DROPOUT,
            "STATIC_CATEGORICAL": STATIC_CATEGORICAL, "STATIC_NUMERIC": STATIC_NUMERIC,
            "KNOWN_CATEGORICAL": KNOWN_CATEGORICAL, "KNOWN_NUMERIC": KNOWN_NUMERIC,
            "OBSERVED_CATEGORICAL": OBSERVED_CATEGORICAL, "OBSERVED_NUMERIC": OBSERVED_NUMERIC,
        },
    },
    "models/tft.pt",
)
print("\nmodel saved to models/tft.pt")



MAE:       29.6 sec
Median AE: 20.3 sec
RMSE:      54.0 sec
Bias:      -1.3 sec
R2:        0.5131
within 30s: 65.9%
within 60s: 90.2%
within 120s: 98.0%

model saved to models/tft.pt


## 10. Variable importance (TFT's headline interpretability feature)

Average the Variable Selection Network weights across the test set — this is the thing plain
LSTM/Conv-Attn-LSTM models can't give you: a direct, learned ranking of which static and
time-varying features the model actually relies on. Useful for the discussion/ablation section
of a paper.

In [11]:
from collections import defaultdict

static_weight_sums = defaultdict(float)
temporal_weight_sums = defaultdict(float)
total_static_n = 0
total_temporal_n = 0

model.eval()
with torch.no_grad():
    for static_cat, static_num, known_cat, known_num, obs_cat, obs_num, target, lengths in test_loader:
        static_cat, static_num = static_cat.to(DEVICE), static_num.to(DEVICE)
        known_cat, known_num = known_cat.to(DEVICE), known_num.to(DEVICE)
        obs_cat, obs_num = obs_cat.to(DEVICE), obs_num.to(DEVICE)

        _, static_w, temporal_w, _ = model(
            static_cat, static_num, known_cat, known_num, obs_cat, obs_num, lengths, return_weights=True
        )
        static_w = static_w.cpu().numpy()          # (B, n_static_vars)
        temporal_w = temporal_w.cpu().numpy()      # (B, T, n_temporal_vars)

        static_names = STATIC_CATEGORICAL + STATIC_NUMERIC
        for i, name in enumerate(static_names):
            static_weight_sums[name] += static_w[:, i].sum()
        total_static_n += static_w.shape[0]

        temporal_names = KNOWN_CATEGORICAL + KNOWN_NUMERIC + OBSERVED_CATEGORICAL + OBSERVED_NUMERIC
        mask = (torch.arange(temporal_w.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)).numpy()
        for i, name in enumerate(temporal_names):
            temporal_weight_sums[name] += (temporal_w[:, :, i] * mask).sum()
        total_temporal_n += mask.sum()

print("Static variable importance (avg. selection weight):")
for name, s in sorted(static_weight_sums.items(), key=lambda kv: -kv[1]):
    print(f"  {name:22s} {s / total_static_n:.4f}")

print("\nTemporal variable importance (avg. selection weight):")
for name, s in sorted(temporal_weight_sums.items(), key=lambda kv: -kv[1]):
    print(f"  {name:26s} {s / total_temporal_n:.4f}")


Static variable importance (avg. selection weight):
  shape_id               0.3240
  weekday                0.1624
  month                  0.1211
  service_id             0.1108
  route_id               0.0984
  is_school_day          0.0602
  has_major_event        0.0523
  direction_id           0.0424
  is_weekend             0.0187
  is_federal_holiday     0.0097

Temporal variable importance (avg. selection weight):
  segment_length             0.1856
  scheduled_segment_time     0.0933
  longitude                  0.0755
  latitude                   0.0587
  upstream_delay_seconds     0.0566
  speed_mps                  0.0552
  scheduled_departure        0.0500
  stop_lat                   0.0488
  scheduled_arrival          0.0461
  stop_lon                   0.0460
  bearing                    0.0405
  hour                       0.0297
  scheduled_segment_speed_mps 0.0295
  snowfall_cm                0.0259
  ridership                  0.0233
  headway_seconds            0.0